# Spreadsheets end to end

Loading an `.xlsx` with `read_excel`, working on it as a DataFrame, then
handing the numeric columns to the natural-language layer.

Requires `pip install -e ".[all,dev]"` from the repo root, and a provider key
in `examples/.env` (see the README's Setup section).

In [ ]:
from dotenv import load_dotenv

load_dotenv(".env")

import numpy as np
import numpyai_dashboard as npi

## Loading

`read_excel` returns a `pandas.DataFrame`. Every column is kept, with its type
inferred by the Rust reader.

In [ ]:
df = npi.read_excel("sample_sales.xlsx")
df.head()

In [ ]:
df.dtypes

Text stays text, dates become `datetime64`, `TRUE`/`FALSE` become `bool`, and
blank cells become the null of whichever type the column is. Nothing is dropped.

In [ ]:
df[["discount", "notes"]].isna().sum()

### Options

Pick a sheet by name or index, skip the header row, or read only the first
`n_rows` when you just want to look at the shape of a large file.

In [ ]:
npi.read_excel("sample_sales.xlsx", sheet="Sales", n_rows=5)

## Working in pandas

Ordinary DataFrame work. Note `discount` has blanks, so fill them before
arithmetic.

In [ ]:
df["revenue"] = df["units"] * df["unit_price"] * (1 - df["discount"].fillna(0))

df.groupby("region")["revenue"].sum().sort_values(ascending=False).round(2)

## Handing off to NumPy

`to_numpy()` on numeric columns is effectively free. Pass `columns=` so the
model knows what each column means rather than seeing bare indices.

In [ ]:
cols = ["units", "unit_price", "discount", "revenue"]

arr = npi.array(df[cols].to_numpy(), columns=cols)
arr

## Asking questions

Everything below needs a provider key. Generated code is syntax-checked and
independently judged before a result comes back.

In [ ]:
arr.chat("What is the mean revenue?")

In [ ]:
arr.chat("Correlation between units and revenue.")

Missing values are visible to the model, so it can be asked about them directly.

In [ ]:
arr.chat("How many rows have a missing discount?")

## Several arrays at once

`NumpyAISession` exposes each array to the model as `arr1`, `arr2`, ...

In [ ]:
emea = df.loc[df["region"] == "EMEA", cols].to_numpy()
apac = df.loc[df["region"] == "APAC", cols].to_numpy()

sess = npi.NumpyAISession([emea, apac])
sess.chat("Compare the mean revenue of the two arrays.")

## Diagnosis

Suggests analysis steps for the data rather than computing an answer.

In [ ]:
diag = npi.Diagnosis(sess)
diag.steps(task="Give me 5 steps to analyse this sales data.")

## Verbose mode

`verbose=True` prints every intermediate step: the generated code, the
judgement, and any retries.

In [ ]:
loud = npi.array(df[cols].to_numpy(), columns=cols, verbose=True)
loud.chat("Total revenue where units exceed 40.")